<a href="https://colab.research.google.com/github/kgovardhan409/AgenticAICourse/blob/master/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: May 2026
(Tested on Google Colab)

In [2]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   TAVILY_API_KEY

In [1]:
!pip install -q \
langchain==0.3.14 \
langchain-openai==0.2.14 \
langchain-community==0.3.14 \
openai==1.59.6 \
python-dotenv==1.0.1 \
requests==2.32.3 \
beautifulsoup4==4.12.3 \
wikipedia==1.4.0 \
ipykernel==6.29.5 \
tavily-python==0.5.0


In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [3]:
import requests
from bs4 import BeautifulSoup

In [4]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

Hello, David! How can I help you today?


In [8]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

The capital of the United States is Washington, D.C.


In [9]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

Deep learning is a subfield of artificial intelligence (AI) and machine learning (ML) that focuses on algorithms inspired by the structure and function of the brain, particularly neural networks. With recent advancements in technology and research, deep learning has become a critical area of study. Here’s a list of important topics to learn in deep learning, along with their significance:

1. **Neural Networks Basics**
   - **Importance:** Understanding the architecture and functioning of neural networks is fundamental to grasping how deep learning models process information. Core concepts include neurons, layers, activation functions, and loss functions.

2. **Convolutional Neural Networks (CNNs)**
   - **Importance:** CNNs are pivotal for image and video recognition tasks. Recent advancements in CNN architectures (e.g., ResNet, EfficientNet) have significantly improved performance in computer vision applications, making this topic essential for anyone working in AI-related to visual 

In [10]:
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple custom tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create tool from function
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)

# Tavily Search Tool
tavily_search = TavilySearchResults(max_results=2)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Tools list
tools = [tavily_search, my_tool]

# Create agent
agent = initialize_agent(
    tools,
    llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("What's the weather like today in London?")

print(response)



> Entering new AgentExecutor chain...
I need to find the current weather information for London to answer the question accurately. 
Action: tavily_search_results_json
Action Input: "current weather in London"  
Observation: [{'url': 'https://www.weather25.com/amp/europe/united-kingdom?page=date&date=19-7', 'content': '| Cloud Cover | 25% | 0% | 14% | 100% | 15% | 9% | 9% | 16% |\n| UV Index | 0 | 0 | 0 | 2 | 6 | 6 | 6 | 5 |\n| Visibility | 10 Km | 10 Km | 10 Km | 10 Km | 10 Km | 10 Km | 10 Km | 10 Km | [...] |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- |\n| 00:00 | 03:00 | 06:00 | 09:00 | 12:00 | 15:00 | 18:00 | 21:00 |\n| Temperature | 18°C | 14°C | 13°C | 16°C | 21°C | 25°C | 25°C | 19°C |\n| Weather | Partly cloudy | Clear | Sunny | Overcast | Sunny | Sunny | Sunny | Sunny |\n| Chance of rain | 5% | 6% | 9% | 13% | 1% | 1% | 1% | 2% |\n| Precipitation | 0 mm | 0 mm | 0 mm | 0 mm | 0 mm | 0 mm | 0 mm | 0 mm |\n| Humidity | 62% | 70% | 78% | 56% | 30% | 23% | 22% | 45% |\n| Wind |

In [11]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [12]:
tools = [tavily_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [13]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])



> Entering new AgentExecutor chain...
To answer the question about the invention of the World Wide Web and its impact, I need to gather detailed information on both aspects. I'll start by searching for the inventor and the significance of the World Wide Web. 

Action: tavily_search_results_json  
Action Input: "Who invented the World Wide Web and its impact"  
Observation: [{'url': 'https://brainly.com/question/32497391', 'content': 'Before the Web, accessing the internet required specific technical knowledge and skills. However, the introduction of the World Wide Web in the early 1990s, created by Tim Berners-Lee, made the internet more user-friendly by allowing users to navigate web pages through hyperlinks. This innovation significantly increased the number of internet users across the globe.\n\nFor example, in December 1997, there were 100 million internet users, and by December 2000, this number had ballooned to around 300 million. This rapid growth in users was largely due to t